In [5]:
!pip install evaluate datasets


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


# ---------------------------------------------------------------------------------

In [5]:
import torch
import math
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu
import evaluate
from sklearn.metrics import pairwise_distances

model_path = './models/casual'+ '/checkpoint-24258'
 
model = AutoModelForSeq2SeqLM.from_pretrained(model_path);
model_2 = AutoModelForSeq2SeqLM.from_pretrained('./models/formal'+ '/checkpoint-24258');
tokenizer = AutoTokenizer.from_pretrained(model_path);
 
 
def generate_summary(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
    summary_ids = model2.generate(
        **inputs, 
        max_length=256, 
        num_beams=5, 
        top_p=0.31,
        early_stopping=True,
        do_sample=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)
 
input_text = [
    "Việc chữ sida trùng với tên gọi căn bệnh SIDA AIDS chỉ là ngẫu nhiên .",
    "Tình hình đó buộc McAthur phải ra lệnh cho quân Mỹ và quân Nam Triều Tiên rút lui toàn bộ .",
    "Bệnh dịch này, hễ xâm nhập vào trư tộc, tốc độ lây lan cực kỳ nhanh chóng, không phân biệt chủng loại, một khi nhiễm bệnh, tỷ lệ tử vong đạt đến bách phần bách.",
    "Tại Malaysia, yếu tố sắc tộc có ảnh hưởng đáng kể đến hoạt động chính trị, thể hiện qua việc nhiều chính đảng được xây dựng dựa trên nền tảng dân tộc.",
    "Quân nổi dậy của thằng Castro nó chiếm mẹ thủ đô ngày 3 tháng 1 năm 1959 rồi."
    
]
output_text = [generate_summary(i) for i in input_text]
 

# inputs = tokenizer(input_text, return_tensors="pt", truncation=True)
# labels = tokenizer(output_text, return_tensors="pt", truncation=True).input_ids
# labels[labels == tokenizer.pad_token_id] = -100  # Đánh dấu token padding là -100 để ignore trong tính loss

print(output_text[0])

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


NameError: name 'model2' is not defined

In [24]:
# BLEU
# 1. Đánh giá bằng BLEU
def evaluate_bleu(predictions, references):
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    return bleu.score


bleu = evaluate.load("sacrebleu")
bleu_score = bleu.compute(predictions=output_text, references=[[i] for i in input_text])
print("BLEU:", bleu_score['score'])
print("BLEU (sacrebleu):", evaluate_bleu(output_text, input_text))

BLEU: 35.01556580297482
BLEU (sacrebleu): 35.01556580297482


In [71]:
# ROUGE
# 2. Đánh giá bằng ROUGE
def evaluate_rouge(predictions, references):
    rouge = evaluate.load("rouge")
    results = rouge.compute(predictions=predictions, references=references)
    return results


rouge = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=output_text, references=input_text)
print("ROUGE:", rouge_score)
print("ROUGE (evaluate):", evaluate_rouge(output_text, input_text))

ROUGE: {'rouge1': 0.7704796906627571, 'rouge2': 0.5827684566447127, 'rougeL': 0.6974444015176281, 'rougeLsum': 0.6974444015176281}
ROUGE (evaluate): {'rouge1': 0.7704796906627571, 'rouge2': 0.5827684566447127, 'rougeL': 0.6974444015176281, 'rougeLsum': 0.6974444015176281}


In [5]:
# BERTScore (dùng tiếng Việt pre-trained model nếu có)
import evaluate

# 3. Đánh giá bằng BERTScore
bertscore = evaluate.load("bertscore")
bert_score = bertscore.compute(predictions=output_text, references=input_text, lang="vi")
print("BERTScore (F1):", sum(bert_score['f1']) / len(bert_score['f1']))

BERTScore (F1): 0.8736905217170715


In [65]:
# METEOR
# 4. Đánh giá bằng METEOR
def evaluate_meteor(predictions, references):
    meteor = evaluate.load("meteor")
    results = meteor.compute(predictions=predictions, references=references)
    return results

print("METEOR (evaluate):", evaluate_meteor(output_text, input_text))

[nltk_data] Downloading package wordnet to /home/hiep/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/hiep/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/hiep/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR (evaluate): {'meteor': 0.6269662819282861}


In [70]:
# 5. Đánh giá Content Preservation (Cosine Similarity)
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

def evaluate_content_preservation(predictions, references):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    pred_embeddings = model.encode(predictions)
    ref_embeddings = model.encode(references)
    cosine_sim = cosine_similarity(pred_embeddings, ref_embeddings)
    avg_similarity = np.mean(cosine_sim)
    return avg_similarity

content_preservation_score = evaluate_content_preservation(output_text, input_text)
print("Content Preservation (Cosine Similarity):", content_preservation_score)

Content Preservation (Cosine Similarity): 0.58248544


In [72]:
# 6. Đánh giá Distinct n-grams
def evaluate_distinct_ngrams(predictions, n=2):
    distinct_ngrams = set()
    for sentence in predictions:
        tokens = sentence.split()  # Phân tách từ trong câu
        n_grams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        distinct_ngrams.update(n_grams)
    return len(distinct_ngrams) / len(predictions)

distinct_2grams = evaluate_distinct_ngrams(output_text, n=2)
print("Distinct 2-grams:", distinct_2grams)

Distinct 2-grams: 22.6


In [ ]:
model.eval()
with torch.no_grad():
    res = model(**inputs, labels=labels)
    loss = res.loss
    
perplexity = math.exp(loss.item())
print(f"Perplexity: {perplexity}")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Perplexity: 1.442964654439944
